In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

In [2]:
tif_path = "../data/raw/FTW_Vietnam/s2_images/window_a/g0_0000000000-0000000000.tif"

with rasterio.open(tif_path) as src:
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bands:", src.count)
    print("CRS:", src.crs)
    print("Transform:", src.transform)
    print("Data type:", src.dtypes)
    print("NoData:", src.nodata)

Width: 256
Height: 256
Bands: 4
CRS: EPSG:4326
Transform: | 0.00, 0.00, 106.08|
| 0.00, 0.00, 20.86|
| 0.00, 0.00, 1.00|
Data type: ('uint16', 'uint16', 'uint16', 'uint16')
NoData: None


In [3]:
ROOT = Path("../data/raw/FTW_Vietnam")

img_dir = ROOT / "s2_images" / "window_a"
sem3_dir = ROOT / "label_masks" / "semantic_3class"
inst_dir = ROOT / "label_masks" / "instance"
parquet_path = ROOT / "chips_vietnam.parquet"

print("=== Dataset folders ===")
print("Image dir exists:", img_dir.exists(), img_dir)
print("Semantic 3-class dir exists:", sem3_dir.exists(), sem3_dir)
print("Instance dir exists:", inst_dir.exists(), inst_dir)
print("Parquet exists:", parquet_path.exists(), parquet_path)
print()

image_paths = sorted(img_dir.glob("*.tif"))
sem3_paths = sorted(sem3_dir.glob("*.tif"))
inst_paths = sorted(inst_dir.glob("*.tif"))

print("=== File counts ===")
print("Images:", len(image_paths))
print("Semantic 3-class masks:", len(sem3_paths))
print("Instance masks:", len(inst_paths))
print()

if not image_paths:
    raise RuntimeError("No image chips found. Check your dataset path.")

sample_name = image_paths[0].name
sample_img = img_dir / sample_name
sample_sem3 = sem3_dir / sample_name
sample_inst = inst_dir / sample_name

print("=== Sample pair ===")
print("Sample filename:", sample_name)
print("Image:", sample_img.exists(), sample_img)
print("Semantic 3-class:", sample_sem3.exists(), sample_sem3)
print("Instance:", sample_inst.exists(), sample_inst)
print()

print("=== Image metadata ===")
with rasterio.open(sample_img) as src:
    img = src.read()
    print("Shape [C,H,W]:", img.shape)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bands:", src.count)
    print("Dtypes:", src.dtypes)
    print("Bands (B02 (blue); B03 (green); B04 (red); B08 (NIR)):", src.descriptions)
    print("CRS:", src.crs)
    print("Transform:", src.transform)
    print("NoData:", src.nodata)
    print("Min:", float(np.nanmin(img)))
    print("Max:", float(np.nanmax(img)))
print()

if sample_sem3.exists():
    print("=== Semantic 3-class mask metadata ===")
    with rasterio.open(sample_sem3) as src:
        mask = src.read(1)
        print("Shape:", mask.shape)
        print("Dtype:", mask.dtype)
        print("Unique values:", np.unique(mask))
        print("Min:", mask.min())
        print("Max:", mask.max())
    print()

if sample_inst.exists():
    print("=== Instance mask metadata ===")
    with rasterio.open(sample_inst) as src:
        inst = src.read(1)
        unique_inst = np.unique(inst)
        print("Shape:", inst.shape)
        print("Dtype:", inst.dtype)
        print("Number of unique IDs:", len(unique_inst))
        print("First 20 unique values:", unique_inst[:20])
        print("Min:", inst.min())
        print("Max:", inst.max())
    print()

if parquet_path.exists():
    print("=== chips_vietnam.parquet ===")
    df = pd.read_parquet(parquet_path)
    print("Columns:", list(df.columns))
    print(df.head())

=== Dataset folders ===
Image dir exists: True ../data/raw/FTW_Vietnam/s2_images/window_a
Semantic 3-class dir exists: True ../data/raw/FTW_Vietnam/label_masks/semantic_3class
Instance dir exists: True ../data/raw/FTW_Vietnam/label_masks/instance
Parquet exists: True ../data/raw/FTW_Vietnam/chips_vietnam.parquet

=== File counts ===
Images: 287
Semantic 3-class masks: 287
Instance masks: 287

=== Sample pair ===
Sample filename: g0_0000000000-0000000000.tif
Image: True ../data/raw/FTW_Vietnam/s2_images/window_a/g0_0000000000-0000000000.tif
Semantic 3-class: True ../data/raw/FTW_Vietnam/label_masks/semantic_3class/g0_0000000000-0000000000.tif
Instance: True ../data/raw/FTW_Vietnam/label_masks/instance/g0_0000000000-0000000000.tif

=== Image metadata ===
Shape [C,H,W]: (4, 256, 256)
Width: 256
Height: 256
Bands: 4
Dtypes: ('uint16', 'uint16', 'uint16', 'uint16')
Bands (B02 (blue); B03 (green); B04 (red); B08 (NIR)): ('B04', 'B03', 'B02', 'B08')
CRS: EPSG:4326
Transform: | 0.00, 0.00, 106

From the sample image metadata, we can infer that the bands are:

- B04 = Red
- B03 = Green
- B02 = Blue
- B08 = Near Infrared

Check the Sentinel 2 website for more details about the bands:

https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/bands

In [4]:
df = pd.read_parquet("../data/raw/FTW_Vietnam/chips_vietnam.parquet")

In [5]:
df.info

<bound method DataFrame.info of                         aoi_id  split  \
0    g42_0000000000-0000000000  train   
1    g42_0000000000-0000000512  train   
2    g42_0000000000-0000001024  train   
3    g42_0000000512-0000000000  train   
4    g42_0000000512-0000000512  train   
..                         ...    ...   
295  g51_0000000512-0000000512    val   
296  g51_0000000512-0000001024    val   
297  g51_0000001024-0000000000    val   
298  g51_0000001024-0000000512    val   
299  g51_0000001024-0000001024    val   

                                              geometry  
0    b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...  
1    b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...  
2    b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...  
3    b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...  
4    b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...  
..                                                 ...  
295  b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...  
296  b'\x01\x03\x00

In [6]:
df.head(8)

,aoi_id,split,geometry
0,g42_0000000000-0000000000,train,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
1,g42_0000000000-0000000512,train,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
2,g42_0000000000-0000001024,train,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
3,g42_0000000512-0000000000,train,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
4,g42_0000000512-0000000512,train,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
5,g42_0000000512-0000001024,train,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
6,g42_0000001024-0000000000,train,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
7,g42_0000001024-0000000512,train,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
